# 🔍 EDA & Baseline Analysis

**Цель:** понять данные, выявить паттерны, сформулировать гипотезы фич

**Автор:** Антон Лысогор

**Дата:** 2026-05-01

## Импорт библиотек и загрузка данных

In [ ]:
# 📦 Импорт библиотек
import os
import sys
from pathlib import Path

import pandas as pd

In [ ]:
# Проверка, что используется именно окружение проекта
# Пть должен быть внури проекта ~/ml-hackathon/.venv/bin/python
print(f"Путь к интерпретатору: {sys.executable}")

In [ ]:
# 🗄️ Загрузка данных

#  Адаптация путей под проект
#
# Всегда вычисляем корень относительно ПАПКИ ноутбука,
# а не относительно того, где мы сейчас «гуляем» через chdir
# Если ноутбук в notebooks/eda, то корень — это два уровня вверх
ROOT = Path(os.getcwd())

# Проверка: если мы уже в корне или ушли не туда,
# этот блок поможет не подняться выше проекта (ml-hackathon)
if "notebooks" in str(ROOT):
    ROOT = ROOT.parent.parent

DATA_ROOT = ROOT / "data" / "train"

print(f"Рабочий корень ROOT: {ROOT}")
print(f"Файлы будут искаться в DATA_ROOT: {DATA_ROOT}")

# DATA_ROOT = Path('data/train')
user = pd.read_csv(DATA_ROOT / "user.csv")
shift = pd.read_csv(DATA_ROOT / "shift.csv")
event = pd.read_csv(DATA_ROOT / "event.csv")

print(f"✅ Loaded: users={len(user)}, shifts={len(shift)}, events={len(event)}")

## Первичный обзор данных

In [ ]:
user

In [ ]:
user.info()

In [ ]:
df = user
summary = pd.DataFrame({"dtype": df.dtypes, "unique_count": df.nunique()})
print(summary)

Таблица ***user*** — всё нормально, пропущенных нет, поле *id* — все значения уникальны, числовые и булевые значения в соответствующих форматах.

In [ ]:
shift

In [ ]:
shift.info()

In [ ]:
df = shift
summary = pd.DataFrame({"dtype": df.dtypes, "unique_count": df.nunique()})
print(summary)

In [ ]:
df.describe()

Таблица ***shift***, или *рабочая смена*. Пропущенных значений нет, все id уникальны.

* *start_at* - начало смены,  время в строковой переменной, надо переводить во время или убедиться, что оно в дальнейшем переводится в него.
* 212 локаций, 27 типов работ, 50 заказчиков.
* *workplace_id*: Идентификатор конкретного рабочего места внутри локации. **Примечение:** пока непонятно, что значит.
* *need_mk* — начличие медицинской книжки, связано с полем *has_mk* в таблице *user*.
* *id_differential* — машинный перевод: "Вероятно, признак того, отличается ли оплата в этой смене от стандартной (например, повышенная ставка)." **Примечение:** пока непонятно, что значит.
* *hours* — длительность смены, от 1 часа до 22, средняя 10 часов с отклонением в 3 часа, медианное значение 12 часов.
* *reward* — вознаграждение за смену от 125 до 9500. 125 - низкое значение, стоит проверить. Середина и среднее в районе 3000. Из этого поля можем получить расчётную ставку за один час. Чем она больше, тем больше пользователь будет мотивирован выйти на смену.
* *capacity* — количество человек на смену. Обычно 1, максимум 154, похоже на выброс.

In [ ]:
event

In [ ]:
event.info()

In [ ]:
df = event
summary = pd.DataFrame({"dtype": df.dtypes, "unique_count": df.nunique()})
print(summary)

In [ ]:
df.interaction.value_counts()

In [ ]:
df.ts.value_counts()

Таблица ***event***, пожалуй, самая важная. по видимому представляет лог действий. Пользователь, посмотрел VIEW, при этом может смотреть несколько раз; записался  APPLY и либо вышел FINISHED, либо отказался USER_CANCEL. Ещё есть вариант SYSTEM_CANCEL — смена была отменена.

Пропущенных значений нет.

* *shift_id* — 30k, хотя в таблице *shift* их 39k.
* *user_id* — 3k, а в таблице *user* их 5k. Похоже, что не все смены и все пользователи были активны.
* *ts*, time stamp - в строковом виде.

## Вывод

В целом все данные корректны, только дата тербует преобразования. 

**Дальнейшие шаги:** перед преобразованием и наращиванием фич, теперь надо изучить baseline, что конкретно там уже сделано.